# 06 Mass continuity per lobe

Ice flux through the three flux gates (ITS_LIVE + bed + DEMs), flux divergence, mean elevation change and surface mass balance
for the whole lobe and the Agassiz, Seward and Marvine tributary lobes (Table 4, Fig. 7), for the four DEM intervals and directly for 1999-2022.
Then the SMB of clean ice (non-clean cover assumed stagnant) and the cumulative Agassiz influx over 1999-2002.

Writes `../Data_Paper/vectors/lobe_mask_downstream_of_gates.tif` and `../Data_Paper/tables/mass_continuity/`:
`fluxes_gates.csv`, `cmb_lobe.csv`, `cmb_tributaries.csv`, `smb_by_surface_cover.csv`, `cleanice_smb.csv`, `agassiz_influx_1999_2002.csv`.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import pyproj
import geoutils as gu
import xdem
from rasterio.transform import from_origin
from shapely.geometry import LineString
from scipy import ndimage
from tqdm import tqdm

OUT = '../Data_Paper/tables/mass_continuity/'
os.makedirs(OUT, exist_ok=True)

## DEMs, stable terrain and DEM uncertainty
Loads the five DEMs on the 2019 grid, builds the stable-terrain mask and estimates each DEM's uncertainty (xDEM, 2002 and 2012 sharing the error equally).

In [ ]:
dems = {}

dems[2019] = xdem.DEM('../Data_Paper/DEMs/d2019_icesat2.tif')
dems[2019].set_nodata(np.nan)
dems[2022] = xdem.DEM('../Data_Paper/DEMs/d2022_icesat2.tif').reproject(dems[2019])
dems[2022].set_nodata(np.nan)
dems[2012] = xdem.DEM('../Data_Paper/DEMs/d2012_coregistered.tif').reproject(dems[2019])
dems[2012].set_nodata(np.nan)
dems[1999] = xdem.DEM('../Data_Paper/DEMs/d1999_coregistered.tif').reproject(dems[2019])
dems[1999].set_nodata(np.nan)
dems[2002] = xdem.DEM('../Data_Paper/DEMs/d2002_coregistered.tif').reproject(dems[2019])
dems[2002].set_nodata(np.nan)

# For each entry, set values <=0 to 0
for year in dems:
    dems[year].data[dems[year].data <= 0] = np.nan

glacier_outlines = gu.Vector('../External_Data/RGI60/01_rgi60_Alaska.shp')

stable_terrain = ~glacier_outlines.create_mask(dems[2012])
# Remove terrain where slope > 20 degrees, where dem is below 100m (remove trees) and above 1500m (remove unstable slopes), and where 2012-2002 is too big (glaciers unacounted for)
slope = dems[2012].slope()
slope_mask = slope.data < 20
# Combine the inlier mask and slope mask
stable_terrain.data = stable_terrain.data & slope_mask & (dems[2012].data > 100) & (dems[2012].data < 1500)
# Remove invalid entries from the inlier mask
stable_terrain.data[dems[2012].data.mask==True]=False
stable_terrain.data[dems[2002].data.mask==True]=False
stable_terrain.data = np.logical_and(stable_terrain.data,np.abs((dems[2012].data.data - dems[2002].data.data) < 5))

dates_dems = [pd.to_datetime('1999-09-01'),
              pd.to_datetime('2002-08-04'),
              pd.to_datetime('2012-08-30'),
              pd.to_datetime('2019-10-10'),
              pd.to_datetime('2022-09-18')]

time_intervals = []
# Calculate the time intervals
time_intervals.append((pd.to_datetime('2002-08-04') - pd.to_datetime('1999-09-01')).days / 365.25)
time_intervals.append((pd.to_datetime('2012-08-30') - pd.to_datetime('2002-08-04')).days / 365.25)
time_intervals.append((pd.to_datetime('2019-10-10') - pd.to_datetime('2012-08-30')).days / 365.25)
time_intervals.append((pd.to_datetime('2022-09-18') - pd.to_datetime('2019-10-10')).days / 365.25)
time_intervals.append((pd.to_datetime('2022-09-18') - pd.to_datetime('1999-09-01')).days / 365.25)

dems_sig = {}
years_unc = list(dems.keys())
years_unc = sorted(years_unc)

for i, year in enumerate(years_unc):
    if year == 2012:
        dems_sig[year] = dems[year].estimate_uncertainty(dems[2002], stable_terrain = stable_terrain, precision_of_other='same', random_state=42)  # [v16] fixed seed: unseeded fit failed to converge once
    else:
        dems_sig[year] = dems[year].estimate_uncertainty(dems[2012], stable_terrain = stable_terrain, precision_of_other='finer', random_state=42)  # [v16] fixed seed: unseeded fit failed to converge once

## dh/dt, uncertainties and variograms (from step 02)
`calculate_area_mean_and_error`: area-mean dh/dt and its uncertainty, with the number of effective samples from the variogram model.

In [ ]:
# dh/dt, σ(dh/dt) and variogram models from 02_elevation_change.ipynb (same period order as there)
periods = ['2002-1999', '2012-2002', '2019-2012', '2022-2019', '2022-1999']
dhdt, dhdt_sig, dhdt_variograms = {}, {}, {}
for period in periods:
    end, start = period.split('-')
    dhdt[period] = gu.Raster(f'../Data_Paper/elevation_change/dhdt_{end}_{start}.tif')
    dhdt_sig[period] = gu.Raster(f'../Data_Paper/elevation_change/dhdt_sig_{end}_{start}.tif')
    dhdt_variograms[period] = pd.read_csv(f'../Data_Paper/elevation_change/{end}_{start}_variogram.csv', index_col=0)


def calculate_area_mean_and_error(dh, sig_dh, area_mask, params_variogram_model, rasterize_resolution = 1000):

    area_vector = gu.Vector(area_mask.polygonize())

    # Calculate the area-averaged uncertainty with these models
    neff = xdem.spatialstats.number_effective_samples(area=area_vector, params_variogram_model=params_variogram_model, rasterize_resolution = rasterize_resolution)

    # Calculate the mean random error in the area
    mean_sig = np.nanmean(sig_dh[area_mask])

    # Compute the standard error
    sig_mean_dh = mean_sig / np.sqrt(neff)

    # Mean elevation difference
    mean_dh = np.nanmean(dh[area_mask])

    return mean_dh, sig_mean_dh


## Tributary outlines
RGI 6.0 outlines of Agassiz, Seward and Marvine, used to split the lobe into tributary lobes.

In [ ]:
# Load the glacier mask
glacier_mask = gu.Vector('../External_Data/RGI60/01_rgi60_Alaska.shp').ds

# Select glaciers of interest using the actual GeoDataFrame (glacier_mask): Seward, Marvine, Agassiz
glaciers_of_interest = glacier_mask[glacier_mask['RGIId'].isin(['RGI60-01.13696', 'RGI60-01.23653', 'RGI60-01.23649'])]

# Individual glacier masks
Agassiz_glacier = glaciers_of_interest[glaciers_of_interest['RGIId'] == 'RGI60-01.23649'].reset_index(drop=True)
Seward_glacier = glaciers_of_interest[glaciers_of_interest['RGIId'] == 'RGI60-01.13696'].reset_index(drop=True)
Marvine_glacier = glaciers_of_interest[glaciers_of_interest['RGIId'] == 'RGI60-01.23653'].reset_index(drop=True)

glaciers_data = {
    'Agassiz': Agassiz_glacier,
    'Seward': Seward_glacier,
    'Marvine': Marvine_glacier
}

## Flux gate functions
ITS_LIVE annual mosaics weighted by the fraction of each year inside the interval, stable-ground velocity bias, and gate flux with its velocity, thickness and k error components (Eq. flux).

In [ ]:
def importer(list_files, sdate, edate):

        list_files = glob.glob('../External_Data/ITS_LIVE/*')
        # Initialize an array that will host the dates
        dates_files = [pd.to_datetime(f"{list_files[i].split('/')[-1].split('_')[-2].split('.')[0]}-06-15") for i in range(len(list_files))]
        inds_sort = np.argsort(dates_files)

        dates_files = [dates_files[i] for i in inds_sort]
        years = [dates_files[i].year for i in range(len(dates_files))]
        list_files = [list_files[i] for i in inds_sort]

        inds_valid = np.where((np.array(years) >= sdate.year) & (np.array(years) <= edate.year))[0]

        # Calculate the timedelta in days between first file and sdate, last file and edate
        tdstart = (pd.to_datetime(f"{years[inds_valid[0]] + 1}-01-01") - sdate).days  # [v16] fraction of first year AFTER sdate
        tdend = (edate - pd.to_datetime(f"{years[inds_valid[-1]]}-01-01")).days  # [v16] fraction of last year BEFORE edate

        # List of variables to drop (we drop everything but the variables written below)
        drop = [ele for ele in list(
                xr.open_dataset(list_files[0]).variables
                ) if ele not in ['x','y','vx','vy','v','v_error','vx_error', 'vy_error']
        ]

        list_files = [list_files[i] for i in inds_valid]
        years = [years[i] for i in inds_valid]
        dates_files = [dates_files[i] for i in inds_valid]
        ratios = np.ones(len(years))
        ratios[0] = tdstart/365.25
        ratios[-1] = tdend/365.25

        return list_files, dates_files, drop, ratios


def Calculate_avg_velocities(file, drop, bounds_3413, bed_vels_mask, bed_flag = False):

    # Load the dataset and reproject it to 3413
    ds = xr.open_dataset(file, drop_variables=drop).sel(
        y=slice(bounds_3413.top, bounds_3413.bottom),
        x=slice(bounds_3413.left, bounds_3413.right)
    )

    ### Prepare the transform to convert from netcdf to xdem format
    # Extract x and y coordinates
    x_coords = ds.x.values
    y_coords = ds.y.values
    # Compute resolution
    x_res = (x_coords[-1] - x_coords[0]) / (len(x_coords) - 1)
    y_res = (y_coords[0] - y_coords[-1]) / (len(y_coords) - 1)  # Negative for top-left origin
    # Create affine transformation
    transform_netcdf = from_origin(x_coords[0], y_coords[0], x_res, y_res)

    # Convert to Raster
    ds_vel = gu.Raster.from_array(data=ds.v.values, crs=3413, transform=transform_netcdf, nodata=np.nan).reproject(crs=32607)
    ds_vx = gu.Raster.from_array(data=ds.vx.values, crs=3413, transform=transform_netcdf, nodata=np.nan).reproject(crs=32607)
    ds_vy = gu.Raster.from_array(data=ds.vy.values, crs=3413, transform=transform_netcdf, nodata=np.nan).reproject(crs=32607)

    stable_terrain_vel = ~glacier_outlines.create_mask(ds_vx)

    bias_vx = np.nanmean(ds_vx.data.data[stable_terrain_vel.data])
    bias_vy = np.nanmean(ds_vy.data.data[stable_terrain_vel.data])

    if bed_flag == False:
        bed_vels_mask = gu.Vector("../Data_Paper/vectors/bed_polygon.shp").create_mask(ds_vel)
        v_avg = np.nanmean(ds_vel.data[bed_vels_mask.data])

        return v_avg, bias_vx, bias_vy, bed_vels_mask
    else:
        v_avg = np.nanmean(ds_vel.data[bed_vels_mask.data])
        return v_avg, bias_vx, bias_vy


def Flux_calculator(ratios, transect, sdate, edate, years, list_files, thickness, bed_error, bias_vx, bias_vy, drop, k=0.9, deltak=0.1):

    # Calculate the deltax and deltay for the flux calculation
    coords = pd.read_csv(f'../Data_Paper/vectors/flux_gate_transects/{transect}.csv')

    # Convert the coordinates from UTM to polar Sterographic
    points = [pyproj.Transformer.from_proj('epsg:32607','epsg:3413',always_xy=True).transform(*(coords.x.values[i], coords.y.values[i])) for i in range(len(coords))]
    points = pd.DataFrame(points, columns=['x','y'])

    deltax = np.diff(points.x.values)
    deltay = np.diff(points.y.values)
    xmid = points.x.values[0:-1] + deltax*0.5
    ymid = points.y.values[0:-1] + deltay*0.5

    # Update the coordinates at which we calculate flux
    points = pd.DataFrame({'x':xmid, 'y':ymid})

    tr = xr.open_dataset(list_files[0],
                        drop_variables=drop
                        ).sel(y=xr.DataArray(points.y.values, dims='z'),
                        x=xr.DataArray(points.x.values, dims='z'),
                        method='nearest'
                        ).interpolate_na(dim='time', method='cubic')

    # Loop through every file, which we will append to the existing dataset
    for f in list_files[1:]:  # [v16] was range(len(list_files[1:])) -> list_files[i]: first file twice, last never

        drop = [ele for ele in list(
                xr.open_dataset(f).variables
                ) if ele not in ['x','y','vx','vy','v', 'vx_error', 'vy_error']
        ]

        dtemp = xr.open_dataset(f,
                                drop_variables=drop).sel(y=xr.DataArray(points.y.values, dims='z'),
                    x=xr.DataArray(points.x.values, dims='z'),
                    method='nearest'
                    ).interpolate_na(dim='time', method='cubic')

        tr = xr.concat((tr,dtemp), 'time')

    # Calculate the average in time of every variable in tr
    da_ratios = xr.DataArray(ratios, dims=['time'], coords={'time': tr.time})
    tr_mean = tr.weighted(da_ratios).mean(dim='time')
    mean_bias_vx = np.nansum(bias_vx * da_ratios) / np.nansum(da_ratios)
    mean_bias_vy = np.nansum(bias_vy * da_ratios) / np.nansum(da_ratios)

    # Extract DEM at the points
    th_points = thickness.reproject(crs=3413).interp_points((tr.x.values, tr.y.values), method='nearest')

    fluxpt = th_points * k * ((tr_mean.vx.values-mean_bias_vx) * deltay - (tr_mean.vy.values-mean_bias_vy) * deltax)

    deltah = bed_error.reproject(crs=3413).interp_points((tr.x.values, tr.y.values), method='nearest')
    # Calculate errors
    evx = ((tr_mean.vx_error)**2) * (th_points * deltay * k)**2
    evy = ((tr_mean.vy_error)**2) * (th_points * deltax * k)**2
    eh = (deltah**2) * (tr_mean.vx.values * deltay * k - tr_mean.vy.values * deltax * k)**2
    ek = (deltak**2) * (th_points * (tr_mean.vx.values * deltay - tr_mean.vy.values * deltax))**2  # [v16] was k instead of H

    errorpt = np.sqrt(evx+evy+eh+ek)

    # [v16] per-point 1-sigma components (m3/a): velocity, thickness, k
    comps = {"v": np.sqrt(evx + evy), "h": np.sqrt(eh), "k": np.sqrt(ek)}
    return fluxpt, errorpt, comps

## Lobe mask and mass continuity (Table 4)
Lobe mask = bed polygon cut at the three gates, restricted to the RGI tributary outlines. For each DEM interval and for 1999-2022:
gate fluxes, then dh/dt, flux divergence and SMB per tributary lobe (fed by its own gate) and for the whole lobe (fed by all three).
Writes `lobe_mask_downstream_of_gates.tif`.

In [ ]:
# [v16] lobe mask = bed_polygon cut at the three fluxgates: gate lines (extended 500 m, buffered 10 m) are burnt in
# as a barrier and only the connected lobe downstream of them is kept -> no dh/dt pixel upstream of a gate.
# Each tributary lobe = RGI outline & lobe_mask, fed only by its own gate; whole lobe = union of the three, fed by all three.
_gate_lines = []
for _g in ['Agassiz', 'Seward', 'Marvine']:
    _c = pd.read_csv(f'../Data_Paper/vectors/flux_gate_transects/{_g}.csv'); _xy = np.c_[_c.x, _c.y]
    _u = (_xy[-1] - _xy[0]) / np.linalg.norm(_xy[-1] - _xy[0])
    _gate_lines.append(LineString(np.vstack([_xy[0] - _u * 500, _xy, _xy[-1] + _u * 500])).buffer(10))
_barrier = gu.Vector(gpd.GeoDataFrame(geometry=_gate_lines, crs=32607)).create_mask(dems[2019]).data
lobe_mask = gu.Vector("../Data_Paper/vectors/bed_polygon.shp").create_mask(dems[2019])
_lab, _n = ndimage.label(lobe_mask.data & ~_barrier)
_sizes = ndimage.sum(np.ones_like(_lab), _lab, index=np.arange(1, _n + 1))
lobe_mask.data = _lab == (np.argmax(_sizes) + 1)
# [v16] sublobes = RGI tributary outlines; whole lobe = their union (margins outside RGI are dropped)
lobe_mask.data = lobe_mask.data & np.any([gu.Vector(glaciers_data[_g]).create_mask(dems[2019]).data for _g in ['Agassiz', 'Seward', 'Marvine']], axis=0)
print(f"lobe mask downstream of gates: {lobe_mask.data.sum() * np.prod(dems[2019].res) / 1e6:.1f} km2")
lobe_mask.save('../Data_Paper/vectors/lobe_mask_downstream_of_gates.tif')

gate_rows, lobe_rows = [], []  # [v16]
dhdt_list = []
dhdt_error_list = []
CMB_list = []
CMB_error_list = []
Fluxdiv_list = []
Fluxdiv_error_list = []

# Get 3413 bounds of the DEMs
bounds_3413 = dems[2019].reproject(crs=3413).bounds

# [v19] the 4 DEM intervals + the direct 1999-2022 balance (dh/dt from 2022-1999, H from mean(1999, 2022), velocities weighted over 1999-2022)
for d0, d in tqdm([(0, 1), (1, 2), (2, 3), (3, 4), (0, 4)]):

    list_files = glob.glob('../External_Data/ITS_LIVE/*')
    sdate, edate = dates_dems[d0], dates_dems[d]
    yr1, yr2 = years_unc[d0], years_unc[d]  # [v16] DEM keys
    period = f"{yr2}-{yr1}"
    list_files, years, drop, ratios = importer(list_files, sdate, edate)

    bias_vx = []
    bias_vy = []
    vel_lobe, bx, by, bed_vel_mask = Calculate_avg_velocities(list_files[0], drop, bounds_3413, None, bed_flag=False)
    bias_vx.append(bx)
    bias_vy.append(by)

    # Then iterate through the files and get the stable-ground velocity bias
    for v in list_files[1:]:
        v_avg, bx, by = Calculate_avg_velocities(v, drop, bounds_3413, bed_vel_mask, bed_flag=True)
        bias_vx.append(bx)
        bias_vy.append(by)

    # Choose ratio factor of surface vel on average vel
    k = 0.95
    deltak = 0.05 #error
    EPSG_ds = '3413'

    list_transects = ['Agassiz','Seward','Marvine']
    fluxpt = []
    errorspt = []
    cmb_trib = []
    cmb_trib_sig = []
    dhdt_trib = []
    dhdt_trib_sig = []
    fluxdiv_trib = []
    fluxdiv_trib_sig = []

    bed = xdem.DEM('../Data_Paper/bed/bed_elevation.tif').reproject(dems[yr1])
    bed_error = xdem.DEM('../Data_Paper/bed/bed_error.tif').reproject(dems[yr1])
    thickness = xdem.DEM.from_array(data=(((dems[yr1].data + dems[yr2].data)/2) - bed.data).data, crs=dems[yr1].crs, transform=dems[yr1].transform, nodata=np.nan)
    bed_mask = lobe_mask  # [v16] was bed_polygon.shp, which reaches up to ~0.5 km upstream of the gates

    thickness_error = np.sqrt(bed_error**2 + (dems_sig[yr1][0]**2 + dems_sig[yr2][0]**2)/4)  # [v16] units fix

    ### We calculate the CMB for each tributary separately, and then we calculate the total CMB

    for i in range(len(list_transects)):
        flux, errors, comps = Flux_calculator(ratios, list_transects[i], sdate, edate, years, list_files, thickness, thickness_error, bias_vx, bias_vy, drop, k=k, deltak=deltak)  # [v16] k = 0.95 +/- 0.05
        fluxpt.append(flux)
        errorspt.append(errors)

        # Make lobe mask for the tributary
        trib_mask = gu.Vector(glaciers_data[list_transects[i]]).create_mask(dhdt[period]) & bed_mask
        mean_dhdt, mean_dhdt_sig = calculate_area_mean_and_error(dhdt[period], dhdt_sig[period], trib_mask, dhdt_variograms[period], rasterize_resolution = 1000)
        area_mask = np.count_nonzero(trib_mask.data) * np.prod(dhdt[period].res)
        flux_div = -np.nansum(flux) # We calculated the convervegence until now
        Q = np.nansum(flux)
        sig_quad = np.sqrt(np.nansum(errors**2))
        sig_lin = np.nansum(errors)
        sv, sh = np.nansum(comps["v"]), np.nansum(comps["h"])  # correlated along the gate
        sig_mixed = np.sqrt(sv**2 + sh**2 + (deltak / k * Q)**2)
        gate_rows.append({"period": period, "gate": list_transects[i], "start": sdate.date(), "end": edate.date(),
                          "years": ",".join(str(y.year) for y in years), "weights": ",".join(f"{r:.3f}" for r in ratios),
                          "Q_km3_a": Q / 1e9, "sig_quad_km3_a": sig_quad / 1e9, "sig_lin_km3_a": sig_lin / 1e9,
                          "sig_mixed_km3_a": sig_mixed / 1e9, "sig_v_km3_a": sv / 1e9, "sig_h_km3_a": sh / 1e9,
                          "sig_k_km3_a": deltak / k * abs(Q) / 1e9, "area_km2": area_mask / 1e6})
        flux_sig = sig_mixed
        CMB = mean_dhdt + flux_div/area_mask
        CMB_sig = np.sqrt(mean_dhdt_sig**2 + (flux_sig/area_mask)**2)

        cmb_trib.append(CMB)
        cmb_trib_sig.append(CMB_sig)
        dhdt_trib.append(mean_dhdt)
        dhdt_trib_sig.append(mean_dhdt_sig)
        fluxdiv_trib.append(flux_div/area_mask)
        fluxdiv_trib_sig.append(flux_sig/area_mask)

    flux_tot = np.nansum([np.sum(flux) for flux in fluxpt[:3]])
    flux_div = -flux_tot # We calculated the convervegence until now
    g = [r for r in gate_rows if r["period"] == period]
    sig_tot = {"quad": np.sqrt(np.nansum([np.nansum(e**2) for e in errorspt[:3]])),
               "lin": np.nansum([np.nansum(e) for e in errorspt[:3]]),
               # mixed: v and h correlated within a gate, independent between gates; k common to all gates
               "mixed": np.sqrt(sum((r["sig_v_km3_a"]*1e9)**2 + (r["sig_h_km3_a"]*1e9)**2 for r in g) + (deltak / k * flux_tot)**2)}
    flux_sig = sig_tot["mixed"]

    mean_dhdt, mean_dhdt_sig = calculate_area_mean_and_error(dhdt[period], dhdt_sig[period], bed_mask, dhdt_variograms[period], rasterize_resolution = 1000)

    area_mask = np.count_nonzero(bed_mask.data) * np.prod(dhdt[period].res)

    CMB = mean_dhdt + flux_div/area_mask
    CMB_sig = np.sqrt(mean_dhdt_sig**2 + (flux_sig/area_mask)**2)

    lobe_rows.append({"period": period, "area_km2": area_mask / 1e6, "Q_in_km3_a": flux_tot / 1e9,
                      "dhdt_m_a": mean_dhdt, "sig_dhdt_m_a": mean_dhdt_sig, "fluxdiv_m_a": flux_div / area_mask,
                      **{f"sig_fluxdiv_{s}_m_a": v / area_mask for s, v in sig_tot.items()}, "CMB_m_a": CMB,
                      **{f"sig_CMB_{s}_m_a": np.sqrt(mean_dhdt_sig**2 + (v / area_mask)**2) for s, v in sig_tot.items()}})
    print(f"For {yr2}-{yr1}: CMB = {CMB:.2f} ± {CMB_sig:.2f}, Flux Div = {flux_div/area_mask:.2f} ± {flux_sig/area_mask:.2f}, dhdt = {mean_dhdt:.2f} ± {mean_dhdt_sig:.2f}")

    cmb_trib = [CMB] + cmb_trib
    cmb_trib_sig = [CMB_sig] + cmb_trib_sig
    dhdt_trib = [mean_dhdt] + dhdt_trib
    dhdt_trib_sig = [mean_dhdt_sig] + dhdt_trib_sig
    fluxdiv_trib = [flux_div/area_mask] + fluxdiv_trib
    fluxdiv_trib_sig = [flux_sig/area_mask] + fluxdiv_trib_sig

    # Append the main results
    dhdt_list.append(dhdt_trib)
    dhdt_error_list.append(dhdt_trib_sig)
    CMB_list.append(cmb_trib)
    CMB_error_list.append(cmb_trib_sig)
    Fluxdiv_list.append(fluxdiv_trib)
    Fluxdiv_error_list.append(fluxdiv_trib_sig)

## Save fluxes and continuity results
Writes `fluxes_gates.csv` (per gate and period), `cmb_lobe.csv` (whole lobe, all uncertainty schemes) and `cmb_tributaries.csv` (Table 4, Fig. 7; mixed scheme).

In [ ]:
# [v16] save fluxes and continuity results
gates = pd.DataFrame(gate_rows)
trib = []
for p, per in enumerate(dhdt_list):
    period = lobe_rows[p]["period"]
    for j, name in enumerate(['Lobe', 'Agassiz', 'Seward', 'Marvine']):
        trib.append({"period": period, "unit": name, "dhdt_m_a": dhdt_list[p][j], "sig_dhdt_m_a": dhdt_error_list[p][j],
                     "fluxdiv_m_a": Fluxdiv_list[p][j], "sig_fluxdiv_mixed_m_a": Fluxdiv_error_list[p][j],
                     "CMB_m_a": CMB_list[p][j], "sig_CMB_mixed_m_a": CMB_error_list[p][j]})
lobe = pd.DataFrame(lobe_rows)
trib = pd.DataFrame(trib)
gates.to_csv(OUT + 'fluxes_gates.csv', index=False); lobe.to_csv(OUT + 'cmb_lobe.csv', index=False); trib.to_csv(OUT + 'cmb_tributaries.csv', index=False)

## SMB by surface cover, all inflow to clean ice
Debris (> -5 m/a), melt features/calving (debris <= -5 m/a) and vegetation-covered ice assumed stagnant; all gate inflow goes to clean ice:
SMB_clean = dhdt_clean - Q_in / A_clean. Writes `smb_by_surface_cover.csv`.

In [ ]:
# [v20] clean-ice SMB: debris (> -5 m/a), thermokarst/calving (debris <= -5 m/a, per period) and vegetation are assumed
# stagnant (SMB = dh/dt); all gate inflow goes to clean ice: SMB_clean = dhdt_clean - Q_in / A_clean
SHP = '../Data_Paper/vectors/'
cover_vec = {'Clean ice': 'Area_cleanice.shp', 'Debris-covered ice': 'Area_moraines.shp', 'Vegetation-covered ice': 'Area_forelands.shp'}
cover_static = {c: gu.Vector(SHP + f).create_mask(dems[2019]).data for c, f in cover_vec.items()}
px = np.prod(dems[2019].res)
units = {g: gu.Vector(glaciers_data[g]).create_mask(dems[2019]).data & lobe_mask.data for g in ['Agassiz', 'Seward', 'Marvine']}
units['Lobe'] = lobe_mask.data.copy()
flux = {}  # (period, unit) -> (Q m3/a, sigma_Q m3/a), mixed scheme
for r in gate_rows:
    flux[(r['period'], r['gate'])] = (r['Q_km3_a'] * 1e9, r['sig_mixed_km3_a'] * 1e9)
for r in lobe_rows:
    flux[(r['period'], 'Lobe')] = (r['Q_in_km3_a'] * 1e9, r['sig_fluxdiv_mixed_m_a'] * r['area_km2'] * 1e6)

def mk(arr):
    m = lobe_mask.copy(); m.data = arr; return m

clean_rows = []
for period in ['2002-1999', '2012-2002', '2019-2012', '2022-2019', '2022-1999']:
    dd = dhdt[period].data.filled(np.nan)
    classes = {'Clean ice': cover_static['Clean ice'],
               'Debris (> -5)': cover_static['Debris-covered ice'] & ~(dd <= -5),
               'Thermo./Calving': cover_static['Debris-covered ice'] & (dd <= -5),
               'Vegetation-covered ice': cover_static['Vegetation-covered ice']}
    for u, um in units.items():
        row = {'period': period, 'unit': u, 'A_unit_km2': um.sum() * px / 1e6}
        covered = np.zeros_like(um)
        for c, cm in classes.items():
            m = um & cm & ~covered          # first class wins where shapefiles overlap
            covered |= m
            if m.sum() == 0:
                row[f'A_{c}_km2'] = 0.0; row[f'dhdt_{c}'] = np.nan; row[f'sig_{c}'] = np.nan; continue
            mu, sg = calculate_area_mean_and_error(dhdt[period], dhdt_sig[period], mk(m), dhdt_variograms[period], rasterize_resolution=1000)
            row[f'A_{c}_km2'] = m.sum() * px / 1e6; row[f'dhdt_{c}'] = float(mu); row[f'sig_{c}'] = float(sg)
        row['A_unclassified_km2'] = (um & ~covered).sum() * px / 1e6
        Q, sQ = flux[(period, u)]
        Aw = row['A_Clean ice_km2'] * 1e6
        row['Q_in_km3_a'] = Q / 1e9
        row['fluxdiv_clean_m_a'] = -Q / Aw
        row['SMB_clean_m_a'] = row['dhdt_Clean ice'] - Q / Aw
        row['sig_SMB_clean_m_a'] = np.hypot(row['sig_Clean ice'], sQ / Aw)
        clean_rows.append(row)
wdf = pd.DataFrame(clean_rows)
wdf.to_csv(OUT + 'smb_by_surface_cover.csv', index=False)

## Clean-ice SMB partition
Classes inside each lobe: clean ice (c), debris outside melt features (d), melt features (mf), vegetation (v), unclassified (u).
Non-clean classes assumed stagnant: b_c = (b_L A_L - sum_k dh_k A_k) / A_c. Writes `cleanice_smb.csv`.

In [ ]:
# clean-ice SMB per unit. Classes inside each lobe polygon: clean ice (c), debris outside melt features
# (d: debris & dh/dt > -5), melt features (mf: debris & dh/dt <= -5, from the period's own dh/dt), vegetation (v),
# unclassified (u: in none of the three shapefiles). Overlaps: first class wins in the order c, d/mf, v (as v20).
# Non-clean classes are assumed stagnant (b_k = dh_k). b_c = (b_L A_L - sum_k dh_k A_k) / A_c ; b_c' = dh_c + (div q)_L A_L / A_c
tab2 = {}
for p, per in enumerate(dhdt_list):
    period = lobe_rows[p]['period']
    for j, name in enumerate(['Lobe', 'Agassiz', 'Seward', 'Marvine']):
        tab2[(period, name)] = dict(dh=dhdt_list[p][j], s_dh=dhdt_error_list[p][j], div=Fluxdiv_list[p][j], s_div=Fluxdiv_error_list[p][j],
                                    b=CMB_list[p][j], s_b=CMB_error_list[p][j])

def class_stats(period, m):
    if m.sum() == 0:
        return np.nan, np.nan, False
    mu, sg = calculate_area_mean_and_error(dhdt[period], dhdt_sig[period], mk(m), dhdt_variograms[period], rasterize_resolution=1000)
    fb = False
    if not np.isfinite(sg):  # n_eff failed (very small/fragmented class): fall back to mean pixel sigma, no averaging
        sg = float(np.nanmean(dhdt_sig[period][mk(m)])); fb = True
    return float(mu), float(sg), fb

ci_rows = []
for period in ['2002-1999', '2012-2002', '2019-2012', '2022-2019', '2022-1999']:
    dd = dhdt[period].data.filled(np.nan)
    classes = {'c': cover_static['Clean ice'],
               'd': cover_static['Debris-covered ice'] & ~(dd <= -5),
               'mf': cover_static['Debris-covered ice'] & (dd <= -5),
               'v': cover_static['Vegetation-covered ice']}
    for u, um in units.items():
        T = tab2[(period, u)]
        row = {'period': period, 'unit': u, 'A_L_km2': um.sum() * px / 1e6}
        covered = np.zeros_like(um); fallback = []
        masks = {}
        for c, cm in classes.items():
            masks[c] = um & cm & ~covered; covered |= masks[c]
        masks['u'] = um & ~covered
        for c, m in masks.items():
            mu, sg, fb = class_stats(period, m)
            if fb: fallback.append(c)
            row[f'A_{c}_km2'] = m.sum() * px / 1e6; row[f'A_{c}_pct'] = 100 * m.sum() / um.sum()
            row[f'dh_{c}'] = mu; row[f'sig_dh_{c}'] = sg
        row['sigma_fallback_classes'] = ','.join(fallback)
        A = {c: row[f'A_{c}_km2'] * 1e6 for c in masks}; AL = row['A_L_km2'] * 1e6
        dh = {c: (row[f'dh_{c}'] if A[c] > 0 else 0.0) for c in masks}
        sd = {c: (row[f'sig_dh_{c}'] if A[c] > 0 else 0.0) for c in masks}
        row.update(dh_L_tab2=T['dh'], sig_dh_L_tab2=T['s_dh'], divq_L=T['div'], sig_divq_L=T['s_div'], b_L=T['b'], sig_b_L=T['s_b'])
        row['dh_L_classes'] = sum(dh[c] * A[c] for c in masks) / AL
        row['closure_diff'] = row['dh_L_classes'] - T['dh']
        stag = ['d', 'mf', 'v', 'u']
        row['b_c'] = (T['b'] * AL - sum(dh[k] * A[k] for k in stag)) / A['c']
        row['sig_b_c_quad'] = np.sqrt((T['s_b'] * AL) ** 2 + sum((sd[k] * A[k]) ** 2 for k in stag)) / A['c']
        row['sig_b_c_lin'] = (T['s_b'] * AL + sum(sd[k] * A[k] for k in stag)) / A['c']
        row['b_c_prime'] = dh['c'] + T['div'] * AL / A['c']
        row['sig_b_c_prime_quad'] = np.hypot(sd['c'], T['s_div'] * AL / A['c'])
        row['sig_b_c_prime_lin'] = sd['c'] + T['s_div'] * AL / A['c']
        Acu = A['c'] + A['u']  # sensitivity: unclassified counted as clean ice
        row['b_c_u_as_clean'] = (T['b'] * AL - sum(dh[k] * A[k] for k in ['d', 'mf', 'v'])) / Acu
        row['sig_b_c_u_as_clean_quad'] = np.sqrt((T['s_b'] * AL) ** 2 + sum((sd[k] * A[k]) ** 2 for k in ['d', 'mf', 'v'])) / Acu
        row['ratio_b_c_over_dh_d'] = row['b_c'] / dh['d']
        row['flag_Ac_lt_5pct'] = row['A_c_pct'] < 5
        ci_rows.append(row)
cidf = pd.DataFrame(ci_rows)
cidf.to_csv(OUT + 'cleanice_smb.csv', index=False)

## Cumulative Agassiz influx 1999-2002
Agassiz gate volume over the 1999-09-01 to 2002-08-04 DEM interval. Writes `agassiz_influx_1999_2002.csv`.

In [ ]:
# [v21] cumulative Agassiz gate volume over the exact 1999-09-01 -> 2002-08-04 DEM interval (mixed sigma)
g = [r for r in gate_rows if r['period'] == '2002-1999' and r['gate'] == 'Agassiz'][0]
dt = (pd.Timestamp(g['end']) - pd.Timestamp(g['start'])).days / 365.25
vol, svol = g['Q_km3_a'] * dt, g['sig_mixed_km3_a'] * dt
agz = pd.DataFrame([{'start': g['start'], 'end': g['end'], 'dt_a': dt, 'time_intervals_0_a': time_intervals[0], 'Q_km3_a': g['Q_km3_a'],
                     'sig_mixed_km3_a': g['sig_mixed_km3_a'], 'sig_quad_km3_a': g['sig_quad_km3_a'], 'sig_lin_km3_a': g['sig_lin_km3_a'],
                     'V_km3': vol, 'sig_V_mixed_km3': svol, 'sig_V_quad_km3': g['sig_quad_km3_a'] * dt, 'sig_V_lin_km3': g['sig_lin_km3_a'] * dt}])
agz.to_csv(OUT + 'agassiz_influx_1999_2002.csv', index=False)